In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import os, time
import numpy as np

from reflect.llm.prompter import PortkeyLLMPrompter
from reflect.core.utils import load_real_episode
from reflect.models.intrinsics import real_k
from reflect.pipelines.perception.discovery_layer.startup import describe_environment
from reflect.pipelines.perception.process_layer.detect_track import DetectorTracker
from reflect.pipelines.perception.process_layer.reid_embed import MaskedCLIPEncoder
from reflect.pipelines.perception.process_layer.tracks import TrackStore
from reflect.pipelines.perception.process_layer.lift3d import lift_mask
from reflect.pipelines.perception.semantic_layer.geometric_edges import propose_edges
from reflect.pipelines.perception.semantic_layer.llm_edges import normalize_scene
from reflect.pipelines.perception.viz.rerun_stream import RerunStream
from reflect.pipelines.perception.semantic_layer.scene_graph import build_snapshot


In [ ]:
from reflect.core.paths import REPO_ROOT, real_world_data_root

REAL_DATA_ROOT = real_world_data_root()  # honors REFLECT_DATA_ROOT
PORTKEY_API_KEY = os.environ["PORTKEY_API_KEY"]  # set in your shell or .env (see .env.example)

llm = PortkeyLLMPrompter(portkey_api_key=PORTKEY_API_KEY, model="gpt-5.4", reasoning_effort="none")

In [ ]:
EPISODES = [
    ("boilWater1",    5),
]

frames = []
for ep_name, n in EPISODES:
    frames.extend(list(load_real_episode(REAL_DATA_ROOT / ep_name, n)))

print(f"\nTotal frames loaded: {len(frames)}")


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 5))
for i in range(4):
    axes[0, i].imshow(frames[i]["rgb"])
    axes[0, i].set_title(f"Episode: {frames[i]['episode']}, Step: {frames[i]['step']}")
    axes[0, i].axis("off")
    
    depth_vis = frames[i]["depth"]
    axes[1, i].imshow(depth_vis, cmap="plasma")
    axes[1, i].set_title(f"Depth")
    axes[1, i].axis("off")


## Perception - Layer 0 Scene Discovery
Scene uderstanding

In [ ]:
vocab = describe_environment(frames[0]["rgb"])
print("\nCandidate objects in the environment:")
print(vocab)

## Perception - Layer 1 Frame Processing
Process Layer:
Detect (YOLOE) -> Track (Yoloe BotSort) -> Label-Fuse -> 3D Lift -> Store Frame

Instantiate models and trackers.

In [ ]:
YOLOE_WEIGHTS = str(REPO_ROOT / "models" / "yoloe-26x-seg.pt")  # fetched by ultralytics on first use
K = real_k()
detector = DetectorTracker(weights=YOLOE_WEIGHTS, vocab=vocab, conf=0.10)
encoder = MaskedCLIPEncoder()
store = TrackStore(vocab=vocab)

print(f"Detector device: {detector.device}")
print(f"CLIP device:     {encoder.device}")

In [ ]:
# ── Rerun visualisation ─────────────────────────────────────────────────
USE_RERUN = True

# vocab_list is needed inside the frame loop for rerun labels.
vocab_list = list(vocab)

rr_stream = RerunStream(spawn_viewer=True) if USE_RERUN else None


Run frame detection throughout the episodes

In [ ]:
TRACK_CONSOLIDATION_RATE = 20
TIME_LOG_RATE = 100
start_time = time.time()
for i, frame in enumerate(frames):
    depth = frame["depth"]
    rgb = frame["rgb"]
    detections = detector.step(rgb)
    lifted_detections = [lift_mask(depth, det.mask, K) for det in detections]
    store.update(rgb=rgb, detections=detections, lifteds=lifted_detections, step_idx=i, encoder=encoder)

    if i % TRACK_CONSOLIDATION_RATE == 0:
        store.consolidate()

    if rr_stream is not None:
        alive_tracks = store.alive()
        tracks_by_id = {t.track_id: t for t in alive_tracks}
        rr_stream.log_frame(i, rgb, depth, K)
        rr_stream.log_detections(i, detections, tracks_by_id, vocab_list)
        rr_stream.log_tracks_3d(i, alive_tracks, vocab_list)

    if i % TIME_LOG_RATE == 0:
        elapsed = time.time() - start_time
        print(f"Step {i}/{len(frames)} - Elapsed time: {elapsed:.2f}s - Frame Rate: {i/elapsed:.2f} fps")
store.consolidate()


In [ ]:
tracks = list(store.alive())
for track in tracks:
    print(f"Track {track.track_id}: '{track.predicted_label(list(vocab))}' (entropy {track.entropy:.2f}, last seen step {track.last_seen_step})")

In [ ]:
track0 = store.alive()[0] if store.alive() else None
vocab_list = list(vocab)
alpha_values = {
    vocab_list[i]: track0.alpha[i].item() for i in range(len(vocab_list))
}
plt.figure(figsize=(10, 5))
plt.bar(alpha_values.keys(), alpha_values.values())
plt.title(f"Alpha Distribution for Track {track0.track_id} ('{track0.predicted_label(vocab_list)}')")
plt.xlabel("Object Class")
plt.ylabel("Alpha Value")
plt.xticks(rotation=45)
plt.grid(axis="y")
plt.show()

## Perception - Layer 2 Scene Graph

Geometry-based edge computation

In [ ]:
alive = store.alive()
geom_edges = propose_edges(alive)
print(f"Geometric edges over {len(alive)} tracks: {len(geom_edges)}  (verified backbone, source=geometric)")
labels_by_id = {t.track_id: t.predicted_label(vocab_list) for t in alive}
for e in geom_edges:
    src = labels_by_id.get(e.src_track_id, "?")
    dst = labels_by_id.get(e.dst_track_id, "?")
    print(f"  [{e.source:<13}] #{e.src_track_id} ({src})  --{e.relation}-->  #{e.dst_track_id} ({dst})")


LLM normalization

In [ ]:
scene_normalized = normalize_scene(alive, geom_edges, llm, vocab_list)


In [ ]:
spatial = [e for e in scene_normalized.semantic_edges if e.source == "llm_spatial"]
semantic = [e for e in scene_normalized.semantic_edges if e.source == "llm_semantic"]

print(f"== LLMSceneOutput ==")
print(f"  canonical_labels: {len(scene_normalized.canonical_labels)}")
print(f"  semantic_edges:   {len(scene_normalized.semantic_edges)}  "
      f"(llm_spatial={len(spatial)}, llm_semantic={len(semantic)})")
print(f"  merges proposed:  {len(scene_normalized.merges)}\n")

# Canonical labels with diff against current predicted label
by_id = {t.track_id: t for t in alive}
print("Canonical labels:")
for cl in scene_normalized.canonical_labels:
    track = by_id.get(cl.track_id)
    old = track.predicted_label(vocab_list) if track else "<missing>"
    marker = "  ←" if old != cl.label else ""
    print(f"  #{cl.track_id:>5}  {old:<22} -> {cl.label}{marker}")

# Semantic edges, grouped by source
print("\nSemantic edges (gap-fillers + affordances):")
for se in scene_normalized.semantic_edges:
    src = by_id.get(se.src_track_id).predicted_label(vocab_list) if se.src_track_id in by_id else "?"
    dst = by_id.get(se.dst_track_id).predicted_label(vocab_list) if se.dst_track_id in by_id else "?"
    print(f"  [{se.source:<13}] #{se.src_track_id} ({src}) --{se.relation}--> #{se.dst_track_id} ({dst})")
    print(f"      // {se.rationale}")

# Merge proposals
print("\nProposed merges:")
for m in scene_normalized.merges:
    drop = by_id.get(m.drop_track_id)
    keep = by_id.get(m.keep_track_id)
    drop_lab = drop.predicted_label(vocab_list) if drop else "?"
    keep_lab = keep.predicted_label(vocab_list) if keep else "?"
    dist_m = float(np.linalg.norm(drop.last_lifted.centroid - keep.last_lifted.centroid)) if drop and keep else float("nan")
    print(f"  drop #{m.drop_track_id} ({drop_lab})  ->  keep #{m.keep_track_id} ({keep_lab})   d={dist_m*100:.1f}cm")
    print(f"      // {m.rationale}")


In [ ]:
# ── Log final scene graph to rerun ─────────────────────────────────────
if rr_stream is not None:
    snapshot = build_snapshot(
        tracks=alive,
        geom_edges=geom_edges,
        llm_out=scene_normalized,
        step=len(frames) - 1,
        vocab=vocab_list,
    )
    rr_stream.log_scene_graph(len(frames) - 1, snapshot)
    print("Scene graph logged to rerun.")


In [ ]:
import json
print(json.dumps(snapshot.model_dump(), indent=2))